# Density Estimation and Gaussian Mixture Models

This notebook explores density estimation using Mixture of Gaussians (MoG) models trained with the Expectation-Maximisation (EM) algorithm.

The workflow covers:

- exploratory visualisation of vowel formant features
- MoG training for selected phoneme classes
- maximum-likelihood classification using trained density models
- comparison of decision regions for different numbers of Gaussian components
- handling singular covariance matrices through regularisation


In [ ]:
import os
import numpy as np
import scipy
from sklearn.preprocessing import normalize
import matplotlib.pyplot as plt
from IPython import display
%matplotlib inline

## Dataset: Peterson and Barney Vowel Formants

This notebook uses the Peterson and Barney vowel formant dataset. The data contains measurements of the fundamental frequency $F0$ and the first three formant frequencies, $F1$ to $F3$, for sustained English vowels.

The analysis focuses on the $F1$ and $F2$ formants, which are useful acoustic features for distinguishing between vowel sounds.


In [ ]:

from google.colab import files
files.upload()
data_npy_file = 'PB_data.npy'

# Loading data from .npy file
data = np.load(data_npy_file, allow_pickle=True)

In [ ]:
data

## Feature Selection

The dataset contains formant frequency vectors $F0$ to $F3$ and a `phoneme_id` vector representing the ground-truth phoneme class for each sample.

For this analysis, only $F1$ and $F2$ are used as input features. The `phoneme_id` values are used as class labels when evaluating phoneme-specific models and classifiers.


In [ ]:
data = np.ndarray.tolist(data)
# Array that contains the phoneme ID (1-10) of each sample
phoneme_id = data['phoneme_id']

# frequencies f1 and f2
f1 = data['f1']
f2 = data['f2']

## Exploratory Visualisation: $F1$ Against $F2$

The first step is to plot $F1$ against $F2$ to inspect the distribution of the samples and identify whether visible clusters are present in the feature space.


In [ ]:
### your code here


plt.figure(figsize=(10, 10))
plt.scatter(f1, f2, alpha=0.7, c='blue')

plt.xlabel("F1" )
plt.ylabel("F2 ")
plt.title("F1 vs F2")
plt.grid(True)


plt.show()


### Observations

The $F1$ values cover a smaller range than the $F2$ values, with $F2$ showing substantially wider variation across samples.

The scatter plot shows several visible clusters. There is a dense region in the lower-left area of the plot and sparser points toward the upper-right. These overlapping but structured regions suggest that the formant features contain useful information for modelling vowel and phoneme distributions.


# Mixture of Gaussians and the EM Algorithm

A Mixture of Gaussians models a probability distribution as a weighted sum of $K$ Gaussian components. For an observed vector $\mathbf{x}\in\mathbb{R}^D$, the model can be written as:

\begin{equation}
 p(\mathbf{x}) = \sum_{k=1}^K \frac{p(c_{k})}{(2\pi)^{D/2} \mathrm{det}\left(\boldsymbol\Sigma_k\right)^{1/2}}
 \exp\left(-\frac{1}{2}(\mathbf{x}-\boldsymbol\mu_k)^\top {\boldsymbol\Sigma_k}^{-1}(\mathbf{x}-\boldsymbol\mu_k)\right),
\end{equation}

where:

- $\boldsymbol\mu_k\in\mathbb{R}^{D}$ is the mean vector of component $k$
- $\boldsymbol\Sigma_{k}\in\mathbb{R}^{D\times D}$ is the covariance matrix of component $k$
- $p(c_{k})=\pi_k\in\mathbb{R}$ is the mixture coefficient for component $k$

The full set of model parameters is:

\begin{equation}
\theta = \{\boldsymbol\mu_k,\boldsymbol\Sigma_k,\pi_k\}_{k=1}^K
\end{equation}

The EM algorithm alternates between two steps:

- **Expectation step:** estimate the responsibility of each Gaussian component for each data point.
- **Maximisation step:** update the means, covariance matrices and mixture coefficients using those responsibilities.


## Prediction Function Overview

The next section defines the function used to estimate the likelihood of each data point under each Gaussian component.


## E-Step: Component Responsibility Estimation


The `get_predictions` function calculates the likelihood of each data point under each Gaussian component. This corresponds to the expectation step of the EM algorithm, where the model estimates soft assignments between data points and Gaussian components.


In [ ]:
def get_predictions(mu, s, p, X):
	"""
		:param mu			: means of GMM components
		:param s			: covariances of GMM components
		:param p 			: weights of GMM components
		:param X			: 2D array of our dataset
	"""

	# get number of GMM components
	k = s.shape[0]
	# get number of data samples
	N = X.shape[0]
	# get dimensionality of our dataset
	D = X.shape[1]

	Z = np.zeros((N, k))


	for i in range(k):
		#!SECTION - For each gaussian:
		mu_i = mu[i, :] #* Extract mean vector
		mu_i = np.expand_dims(mu_i, axis=1) #* Reshape dimension by expanding
		mu_i_repeated = np.repeat(mu_i, N, axis=1) #* The calculated mean is repeated for other datapoints
		X_minus_mu = X - mu_i_repeated.transpose() #* Transpose
		#!SECTION

		inverse_s = scipy.linalg.pinv(s[i])
		inverse_s = np.squeeze(inverse_s)
		s_i_det = scipy.linalg.det(s[i])
		x_s_x = np.matmul(X_minus_mu, inverse_s)*X_minus_mu
		Z[:, i] = p[i]*(1/np.power(((2*np.pi)**D) * np.abs(s_i_det), 0.5)) * np.exp(-0.5*np.sum(x_s_x, axis=1))

	#*NOTE - Z is a matrix that contains unnormalized likelihood of data point n belonging to it's gaussian
	return Z


## Visualisation Helper Functions


The following helper functions are used to visualise the fitted Gaussian components and, later, the three-dimensional feature representation:

- `plot_gaussians`
- `plot_data_3D`


In [ ]:
def plot_gaussians(ax, s, mu, X, title_string):
    """
        :param X            : 2D array of our dataset
        :param y            : 1D array of the groundtruth labels of the dataset
        :param ax          : existing subplot, to draw on it
    """
    # ax.clear()
    if X.shape[1] == 2:
        # set label of horizontal axis
        ax.set_xlabel('f1')
        # set label of vertical axis
        ax.set_ylabel('f2')
        # scatter the points, with red color
        ax.scatter(X[:,0], X[:,1], c='black', marker='.', label=title_string)

    k = s.shape[0]
    color_list = ['r', 'g', 'b', 'c', 'm', 'y']
    N_sides = 11

    # Iterate over all gaussians
    for k_cnt in range(k):

        # pick the covariance matrix and mean values of each gaussian
        s_k = s[k_cnt]
        mu_k = mu[k_cnt]

        if s_k.shape[0]==2:
            # dataset with 2 features
            side_range = np.arange(0, N_sides, 1)
            theta = side_range/(N_sides-1)*2*np.pi
            matrix = np.array([np.cos(theta), np.sin(theta)])
            mu_k = np.expand_dims(mu_k, axis=1)
            mu_repeated = np.repeat(mu_k, N_sides, axis=1)
            epoints = np.matmul(scipy.linalg.sqrtm(s_k), matrix) + mu_repeated
            plt.plot(epoints[0,:], epoints[1,:], color=color_list[k_cnt], linewidth=2)
            plt.scatter(mu_k[0], mu_k[1], color=color_list[k_cnt], marker='o', linewidth=2)
        elif s_k.shape[0]==3:
            # dataset with 3 features
            side_range = np.arange(0, N_sides, 1)
            theta = side_range/(N_sides-1)*np.pi
            phi = side_range/(N_sides-1)*2*np.pi
            sin_theta = np.expand_dims(np.sin(theta), axis=1)
            cos_theta = np.expand_dims(np.cos(theta), axis=1)
            sin_phi = np.expand_dims(np.sin(phi), axis=1)
            cos_phi = np.expand_dims(np.cos(phi), axis=1)
            sx = np.matmul(sin_theta, cos_phi.transpose())
            sy = np.matmul(sin_theta, sin_phi.transpose())
            sz = np.matmul(cos_theta, np.ones((1, N_sides)))
            svect = np.array([sx.reshape((N_sides*N_sides)), sy.reshape((N_sides*N_sides)), sz.reshape((N_sides*N_sides))])

            mu_k = np.expand_dims(mu_k, axis=1)
            mu_repeated = np.repeat(mu_k, N_sides*N_sides, axis=1)
            epoints = np.matmul(scipy.linalg.sqrtm(s_k), svect) + mu_repeated

            ex = epoints[0]
            ey = epoints[1]
            ez = epoints[2]

            ax.plot3D(epoints[0,:], epoints[1,:], epoints[2,:], color=color_list[k_cnt])
            ax.plot3D(ex, ey, ez, color=color_list[k_cnt])
        else:
            print('Each dataset sample should have either 2 or 3 features. Not plotting this one.')
    ax.legend()
    #! Changed here
    display.clear_output(wait=True)
    display.display(plt.gcf())

#! Not used until question 8
def plot_data_3D(X, title_string, ax):
    """
        :param X            : 2D array of our dataset
        :param y            : 1D array of the groundtruth labels of the dataset
        :param ax          : existing subplot, to draw on it
    """
    # clear subplot from previous (if any) drawn stuff
    #! Changed here
    # ax.clear()
    # set label of x axis
    ax.set_xlabel('f1')
    # set label of y axis
    ax.set_ylabel('f2')
    # set label of z axis
    ax.set_zlabel('f1 + f2')
    # scatter the points, with red color
    ax.scatter3D(X[:,0], X[:,1], X[:,2], c='black', marker='.', label=title_string)
    # add legend to the subplot
    ax.legend()


def save_model(mu, s, p):
  GMM_parameters = {}
  GMM_parameters['mu'] = mu
  GMM_parameters['s'] = s
  GMM_parameters['p'] = p
  if not os.path.isdir('data/'):
    os.mkdir('data/')
  npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
  np.save(npy_filename, GMM_parameters)

## Model Training Workflow

The following code trains a Mixture of Gaussians model using the EM algorithm. The model is trained separately for different phoneme IDs so that each phoneme can be represented by its own density model.


In [ ]:
k = 3 #! Number of guassian components
# id of the phoneme that will be used (e.g. 1, or 2)
p_id = 1 #! Sets phenome
X_full = np.zeros((len(f1), 2))
X_full[:,0] = f1.copy() #! 1st column is F1
X_full[:,1] = f2.copy() #! 2nd column is F2
X_full = X_full.astype(np.float32) #! Convert to float32

X = X_full[phoneme_id==p_id,:] #! Filter by phenome ID, only includes rows with the ID

In [ ]:

#!get number of samples
N = X.shape[0]
#! get dimensionality of our dataset
D = X.shape[1]
#! common practice : GMM weights initially set as 1/k
p = np.ones((k))/k
#! GMM means are picked randomly from data samples
random_indices = np.floor(N*np.random.rand((k)))
random_indices = random_indices.astype(int)
mu = X[random_indices,:] # shape kxD
#! covariance matrices
s = np.zeros((k,D,D)) # shape kxDxD

#! number of iterations for the EM algorithm
n_iter = 100

#! initialize covariances
for i in range(k):
    cov_matrix = np.cov(X.transpose())
    # initially set to fraction of data covariance
    s[i,:,:] = cov_matrix/k

# Initialize array Z that will get the predictions of each Gaussian on each sample
Z = np.zeros((N,k)) # shape Nxk

###############################
# run Expectation Maximization algorithm for n_iter iterations

for t in range(n_iter):
    print('Iteration {:03}/{:03}'.format(t+1, n_iter))

    # !Do the E-step
    Z = get_predictions(mu, s, p, X)
    Z = normalize(Z, axis=1, norm='l1')

    #! Do the M-step:
    for i in range(k):
        mu[i,:] = np.matmul(X.transpose(),Z[:,i]) / np.sum(Z[:,i])

        ###################################################
        # We will fit Gaussians with diagonal covariance matrices:
        mu_i = mu[i,:]
        mu_i = np.expand_dims(mu_i, axis=1)
        mu_i_repeated = np.repeat(mu_i, N, axis=1)
        coef_1 = (X.transpose() - mu_i_repeated)**2
        coef_2 = Z[:,i]/np.sum(Z[:,i])
        coef_2 = np.expand_dims(coef_2, axis=1)
        res_1 = np.squeeze( np.matmul(coef_1, coef_2) )
        s[i,:,:] = np.diag(res_1)

        p[i] = np.mean(Z[:,i])

    fig, ax1 = plt.subplots()
    #! Modified to actually print the iterations...
    title_string = 'Phoneme {} - Iteration {:03}'.format(p_id, t+1)
    plot_gaussians(ax1, 2*s, mu, X, title_string)

#! Made prints pwetty
print('\nFinished.\n')

print("\nGaussian means (mu):\n")
print(f"\n{mu}\n")

print("\nGaussian covariances (s):\n")
print(f"\n{s}\n")

print("\nGuassian mixture weights (p):\n")
print(f"\n{p}\n")
#! Saves the model
save_model(mu, s, p)

## Training a $K=3$ MoG Model for Phoneme 1

This section trains a three-component Mixture of Gaussians model for phoneme 1 and inspects how the fitted components represent the structure of the data.


### Training Behaviour

The Gaussian components are randomly initialised, so their starting positions and shapes can vary between runs. Despite this, repeated training with $K=3$ tends to converge toward a similar structure.

The fitted model usually identifies one clear cluster in the lower-left region, another component in the upper region, and a horizontally elongated component between them. This suggests that the EM algorithm is consistently finding a stable representation of the main density regions in the $F1$-$F2$ feature space.

The means represent the centres of the Gaussian components, while the covariance matrices describe their spread and orientation. Similar final parameter values across repeated runs indicate that the model is converging consistently.


## Training a $K=3$ MoG Model for Phoneme 2


The same training procedure is applied to phoneme 2. This creates a separate class-conditional density model that can later be compared against the phoneme 1 model for classification.


In [ ]:
k = 3 #! Number of guassian components
# id of the phoneme that will be used (e.g. 1, or 2)
p_id = 2 #! Sets phenome
X_full = np.zeros((len(f1), 2))
X_full[:,0] = f1.copy() #! 1st column is F1
X_full[:,1] = f2.copy() #! 2nd column is F2
X_full = X_full.astype(np.float32) #! Convert to float32

X = X_full[phoneme_id==p_id,:] #! Filter by phenome ID, only includes rows with the ID


#!get number of samples
N = X.shape[0]
#! get dimensionality of our dataset
D = X.shape[1]
#! common practice : GMM weights initially set as 1/k
p = np.ones((k))/k
#! GMM means are picked randomly from data samples
random_indices = np.floor(N*np.random.rand((k)))
random_indices = random_indices.astype(int)
mu = X[random_indices,:] # shape kxD
#! covariance matrices
s = np.zeros((k,D,D)) # shape kxDxD

#! number of iterations for the EM algorithm
n_iter = 100

#! initialize covariances
for i in range(k):
    cov_matrix = np.cov(X.transpose())
    # initially set to fraction of data covariance
    s[i,:,:] = cov_matrix/k

# Initialize array Z that will get the predictions of each Gaussian on each sample
Z = np.zeros((N,k)) # shape Nxk

###############################
# run Expectation Maximization algorithm for n_iter iterations

for t in range(n_iter):
    print('Iteration {:03}/{:03}'.format(t+1, n_iter))

    # !Do the E-step
    Z = get_predictions(mu, s, p, X)
    Z = normalize(Z, axis=1, norm='l1')

    #! Do the M-step:
    for i in range(k):
        mu[i,:] = np.matmul(X.transpose(),Z[:,i]) / np.sum(Z[:,i])

        ###################################################
        # We will fit Gaussians with diagonal covariance matrices:
        mu_i = mu[i,:]
        mu_i = np.expand_dims(mu_i, axis=1)
        mu_i_repeated = np.repeat(mu_i, N, axis=1)
        coef_1 = (X.transpose() - mu_i_repeated)**2
        coef_2 = Z[:,i]/np.sum(Z[:,i])
        coef_2 = np.expand_dims(coef_2, axis=1)
        res_1 = np.squeeze( np.matmul(coef_1, coef_2) )
        s[i,:,:] = np.diag(res_1)

        p[i] = np.mean(Z[:,i])

    fig, ax1 = plt.subplots()
    #! Modified to actually print the iterations...
    title_string = 'Phoneme {} - Iteration {:03}'.format(p_id, t+1)
    plot_gaussians(ax1, 2*s, mu, X, title_string)

#! Made prints pwetty
print('\nFinished.\n')

print("\nGaussian means (mu):\n")
print(f"\n{mu}\n")

print("\nGaussian covariances (s):\n")
print(f"\n{s}\n")

print("\nGuassian mixture weights (p):\n")
print(f"\n{p}\n")
#! Saves the model
save_model(mu, s, p)

## Training $K=6$ MoG Models for Both Phonemes

The training process is repeated with six Gaussian components for both phoneme 1 and phoneme 2. Increasing $K$ gives the model more flexibility, but it can also make the fitted density more sensitive to local variation in the data.


### $K=6$ Training for Phoneme 1


In [ ]:
### your code here
k = 6 #! Number of guassian components
# id of the phoneme that will be used (e.g. 1, or 2)
p_id = 1 #! Sets phenome
X_full = np.zeros((len(f1), 2))
X_full[:,0] = f1.copy() #! 1st column is F1
X_full[:,1] = f2.copy() #! 2nd column is F2
X_full = X_full.astype(np.float32) #! Convert to float32

X = X_full[phoneme_id==p_id,:] #! Filter by phenome ID, only includes rows with the ID


#!get number of samples
N = X.shape[0]
#! get dimensionality of our dataset
D = X.shape[1]
#! common practice : GMM weights initially set as 1/k
p = np.ones((k))/k
#! GMM means are picked randomly from data samples
random_indices = np.floor(N*np.random.rand((k)))
random_indices = random_indices.astype(int)
mu = X[random_indices,:] # shape kxD
#! covariance matrices
s = np.zeros((k,D,D)) # shape kxDxD

#! number of iterations for the EM algorithm
n_iter = 100

#! initialize covariances
for i in range(k):
    cov_matrix = np.cov(X.transpose())
    # initially set to fraction of data covariance
    s[i,:,:] = cov_matrix/k

# Initialize array Z that will get the predictions of each Gaussian on each sample
Z = np.zeros((N,k)) # shape Nxk

###############################
# run Expectation Maximization algorithm for n_iter iterations

for t in range(n_iter):
    print('Iteration {:03}/{:03}'.format(t+1, n_iter))

    # !Do the E-step
    Z = get_predictions(mu, s, p, X)
    Z = normalize(Z, axis=1, norm='l1')

    #! Do the M-step:
    for i in range(k):
        mu[i,:] = np.matmul(X.transpose(),Z[:,i]) / np.sum(Z[:,i])

        ###################################################
        # We will fit Gaussians with diagonal covariance matrices:
        mu_i = mu[i,:]
        mu_i = np.expand_dims(mu_i, axis=1)
        mu_i_repeated = np.repeat(mu_i, N, axis=1)
        coef_1 = (X.transpose() - mu_i_repeated)**2
        coef_2 = Z[:,i]/np.sum(Z[:,i])
        coef_2 = np.expand_dims(coef_2, axis=1)
        res_1 = np.squeeze( np.matmul(coef_1, coef_2) )
        s[i,:,:] = np.diag(res_1)

        p[i] = np.mean(Z[:,i])

    fig, ax1 = plt.subplots()
    #! Modified to actually print the iterations...
    title_string = 'Phoneme {} - Iteration {:03}'.format(p_id, t+1)
    plot_gaussians(ax1, 2*s, mu, X, title_string)

#! Made prints pwetty
print('\nFinished.\n')

print("\nGaussian means (mu):\n")
print(f"\n{mu}\n")

print("\nGaussian covariances (s):\n")
print(f"\n{s}\n")

print("\nGuassian mixture weights (p):\n")
print(f"\n{p}\n")
#! Saves the model
save_model(mu, s, p)

### $K=6$ Training for Phoneme 2


In [ ]:
### your code here
k = 6 #! Number of guassian components
# id of the phoneme that will be used (e.g. 1, or 2)
p_id = 2 #! Sets phenome
X_full = np.zeros((len(f1), 2))
X_full[:,0] = f1.copy() #! 1st column is F1
X_full[:,1] = f2.copy() #! 2nd column is F2
X_full = X_full.astype(np.float32) #! Convert to float32

X = X_full[phoneme_id==p_id,:] #! Filter by phenome ID, only includes rows with the ID


#!get number of samples
N = X.shape[0]
#! get dimensionality of our dataset
D = X.shape[1]
#! common practice : GMM weights initially set as 1/k
p = np.ones((k))/k
#! GMM means are picked randomly from data samples
random_indices = np.floor(N*np.random.rand((k)))
random_indices = random_indices.astype(int)
mu = X[random_indices,:] # shape kxD
#! covariance matrices
s = np.zeros((k,D,D)) # shape kxDxD

#! number of iterations for the EM algorithm
n_iter = 100

#! initialize covariances
for i in range(k):
    cov_matrix = np.cov(X.transpose())
    # initially set to fraction of data covariance
    s[i,:,:] = cov_matrix/k

# Initialize array Z that will get the predictions of each Gaussian on each sample
Z = np.zeros((N,k)) # shape Nxk

###############################
# run Expectation Maximization algorithm for n_iter iterations

for t in range(n_iter):
    print('Iteration {:03}/{:03}'.format(t+1, n_iter))

    # !Do the E-step
    Z = get_predictions(mu, s, p, X)
    Z = normalize(Z, axis=1, norm='l1')

    #! Do the M-step:
    for i in range(k):
        mu[i,:] = np.matmul(X.transpose(),Z[:,i]) / np.sum(Z[:,i])

        ###################################################
        # We will fit Gaussians with diagonal covariance matrices:
        mu_i = mu[i,:]
        mu_i = np.expand_dims(mu_i, axis=1)
        mu_i_repeated = np.repeat(mu_i, N, axis=1)
        coef_1 = (X.transpose() - mu_i_repeated)**2
        coef_2 = Z[:,i]/np.sum(Z[:,i])
        coef_2 = np.expand_dims(coef_2, axis=1)
        res_1 = np.squeeze( np.matmul(coef_1, coef_2) )
        s[i,:,:] = np.diag(res_1)

        p[i] = np.mean(Z[:,i])

    fig, ax1 = plt.subplots()
    #! Modified to actually print the iterations...
    title_string = 'Phoneme {} - Iteration {:03}'.format(p_id, t+1)
    plot_gaussians(ax1, 2*s, mu, X, title_string)

#! Made prints pwetty
print('\nFinished.\n')

print("\nGaussian means (mu):\n")
print(f"\n{mu}\n")

print("\nGaussian covariances (s):\n")
print(f"\n{s}\n")

print("\nGuassian mixture weights (p):\n")
print(f"\n{p}\n")
#! Saves the model
save_model(mu, s, p)

## Maximum-Likelihood Classifier Using $K=3$ MoG Models

The two $K=3$ MoG models are used to classify samples as either phoneme 1 or phoneme 2.

For each sample, the classifier calculates the likelihood under both phoneme-specific MoG models:

- $p(\mathbf{x};\boldsymbol\theta_1)$ for phoneme 1
- $p(\mathbf{x};\boldsymbol\theta_2)$ for phoneme 2

The sample is assigned to the phoneme whose model gives the higher likelihood. Classification accuracy and misclassification error are then calculated by comparing the predicted labels with the true `phoneme_id` labels.


In [ ]:
from scipy.stats import multivariate_normal

In [ ]:
k = 3

#^ Get the index corresponding to id = 1 or 2
inds_1 = np.where(phoneme_id == 1)
inds_2 = np.where(phoneme_id == 2)
inds_1 = inds_1[0]
inds_2 = inds_2[0]
inds_1_and_2 = np.concatenate((inds_1, inds_2), axis=0)

X = X_full[inds_1_and_2, :]  # ! This combines data for id = 1 and 2
# print(X.shape[0])
#^ no. of samples
N = X.shape[0]
#^ no. of features
D = X.shape[1]

#! Get true labels
true_labels = phoneme_id[inds_1_and_2]
# print(len(true_labels))
#! FOR ID = 1
# !Get predictions on both phonemes 1 and 2, from a GMM with k components, pretrained on phoneme 1
p_id = 1
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_1 = np.load(npy_filename, allow_pickle=True).item()
    #*NOTE - Don't know what this really contributes to the model. It runs fine without this line from the template. It should already be a dict when pickle=True is used.
    # GMM_parameters_phoneme_1 = np.ndarray.tolist(GMM_parameters_phoneme_1)
else:
    raise('File {} does not exist.'.format(npy_filename))

#! PARAMS FOR 1
mean1 = GMM_parameters_phoneme_1['mu']
s1 = GMM_parameters_phoneme_1['s']
p1 = GMM_parameters_phoneme_1['p']


#! Calculate log-likelihood for phoneme 1
#*NOTE - Loop through each gaussian; calc PDF of gaussian distributions (aka multivariate_normal) with mu and cov params; then sum
likelihood_list_1 = np.zeros(N)
for i in range(k):
    likelihood = multivariate_normal.pdf(X, mean=mean1[i], cov=s1[i])
    likelihood_list_1 = likelihood_list_1 + p1[i] * likelihood







#! FOR ID = 2
# !Get predictions on both phonemes 1 and 2, from a GMM with k components, pretrained on phoneme 2
p_id = 2
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_2 = np.load(npy_filename, allow_pickle=True).item()

else:
    raise('File {} does not exist.'.format(npy_filename))

#! PARAMS FOR 2
mean2 = GMM_parameters_phoneme_2['mu']
s2 = GMM_parameters_phoneme_2['s']
p2 = GMM_parameters_phoneme_2['p']

#! Calc likelihood
#*NOTE - Loop through each gaussian; calc PDF of gaussian distributions (aka multivariate_normal) with mu and cov params; then sum
likelihood_list_2 = np.zeros(N)
for i in range(k):
    likelihood = multivariate_normal.pdf(X, mean=mean2[i], cov=s2[i])
    likelihood_list_2 = likelihood_list_2 + p2[i] * likelihood



# !classify each point using Maximum Likelihood
classified_list = []
for i in range(len(likelihood_list_1)):
    #^ phoneme 1
    if likelihood_list_1[i] > likelihood_list_2[i]:

        classified_list.append(1)
    else:
        #^ phoneme 2
        classified_list.append(2)
classified_list = np.array(classified_list)

# !calculate accuracy
### your code here
correct = np.sum(classified_list == true_labels)
total_samples = len(true_labels)
accuracy = correct / total_samples

missclass_error = 1 - accuracy

print(f"Accuracy: {accuracy:.4f}")
print(f"Misclassification error: {missclass_error:.4f}")


## Classifier Evaluation Using $K=6$ MoG Models

The same maximum-likelihood classification process is repeated using the $K=6$ MoG models. Comparing the $K=3$ and $K=6$ results helps assess whether additional Gaussian components improve classification performance or introduce unnecessary model complexity.


In [ ]:
k = 6

#^ get the index corresponding to id = 1 or 2
inds_1 = np.where(phoneme_id == 1)
inds_2 = np.where(phoneme_id == 2)
inds_1 = inds_1[0]
inds_2 = inds_2[0]
inds_1_and_2 = np.concatenate((inds_1, inds_2), axis=0)

X = X_full[inds_1_and_2, :]  # ! This combines data for id = 1 and 2
# print(X.shape[0])
#^ no. of samples
N = X.shape[0]
#^ no. of features
D = X.shape[1]

#! Get true labels
true_labels = phoneme_id[inds_1_and_2]
# print(len(true_labels))
#! FOR ID = 1
# !Get predictions on both phonemes 1 and 2, from a GMM with k components, pretrained on phoneme 1
p_id = 1
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_1 = np.load(npy_filename, allow_pickle=True).item()
    #*NOTE - Don't know what this really contributes to the model. It runs fine without this line from the template. It should already be a dict when pickle=True is used.
    # GMM_parameters_phoneme_1 = np.ndarray.tolist(GMM_parameters_phoneme_1)
else:
    raise('File {} does not exist.'.format(npy_filename))

#! PARAMS FOR 1
mean1 = GMM_parameters_phoneme_1['mu']
s1 = GMM_parameters_phoneme_1['s']
p1 = GMM_parameters_phoneme_1['p']


#! Calculate log-likelihood for phoneme 1
#*NOTE - Loop through each gaussian; calc PDF of gaussian distributions (aka multivariate_normal) with mu and cov params; then sum
likelihood_list_1 = np.zeros(N)
for i in range(k):
    likelihood = multivariate_normal.pdf(X, mean=mean1[i], cov=s1[i])
    likelihood_list_1 = likelihood_list_1 + p1[i] * likelihood







#! FOR ID = 2
# !Get predictions on both phonemes 1 and 2, from a GMM with k components, pretrained on phoneme 2
p_id = 2
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_2 = np.load(npy_filename, allow_pickle=True).item()

else:
    raise('File {} does not exist.'.format(npy_filename))

#! PARAMS FOR 2
mean2 = GMM_parameters_phoneme_2['mu']
s2 = GMM_parameters_phoneme_2['s']
p2 = GMM_parameters_phoneme_2['p']

#! Calc likelihood
#*NOTE - Loop through each gaussian; calc PDF of gaussian distributions (aka multivariate_normal) with mu and cov params; then sum
likelihood_list_2 = np.zeros(N)
for i in range(k):
    likelihood = multivariate_normal.pdf(X, mean=mean2[i], cov=s2[i])
    likelihood_list_2 = likelihood_list_2 + p2[i] * likelihood



# !classify each point using Maximum Likelihood
classified_list = []
for i in range(len(likelihood_list_1)):
    #^ phoneme 1
    if likelihood_list_1[i] > likelihood_list_2[i]:

        classified_list.append(1)
    else:
        #^ phoneme 2
        classified_list.append(2)


classified_list = np.array(classified_list)

# !calculate accuracy
### your code here
correct = np.sum(classified_list == true_labels)
total_samples = len(true_labels)
accuracy = correct / total_samples

missclass_error = 1 - accuracy

print(f"Accuracy: {accuracy:.4f}")
print(f"Misclassification error: {missclass_error:.4f}")


## Decision Region Visualisation

This section visualises the classifier decision regions across a grid of possible $F1$ and $F2$ values.

A custom grid is created using the minimum and maximum feature values as limits. Each grid point is classified using the trained MoG classifiers, producing a classification matrix where each location is assigned to either phoneme 1 or phoneme 2.

Comparing the $K=3$ and $K=6$ decision regions shows how model complexity affects the smoothness and shape of the classification boundary.


### Decision Regions for $K=3$


In [ ]:

k = 3
# ! Combines for id 1 and 2
X = X_full[inds_1_and_2, :]
# !no. of samples
N = X.shape[0]
# !no. of features
D = X.shape[1]

min_f1 = int(np.min(X[:, 0]))
max_f1 = int(np.max(X[:, 0]))
min_f2 = int(np.min(X[:, 1]))
max_f2 = int(np.max(X[:, 1]))
N_f1 = max_f1 - min_f1
N_f2 = max_f2 - min_f2
print('f1 range: {}-{} | {} points'.format(min_f1, max_f1, N_f1))
print('f2 range: {}-{} | {} points'.format(min_f2, max_f2, N_f2))

#########################################
# Write your code here
# !Create a custom grid of shape N_f1 x N_f2
# !The grid will span all the values of (f1, f2) pairs, between [min_f1, max_f1] on f1 axis, and between [min_f2, max_f2] on f2 axis

f1_values = np.linspace(min_f1, max_f1, N_f1)
f2_values = np.linspace(min_f2, max_f2, N_f2)

#*NOTE -  get two 2d arrays from meshgrid; contains f1_values repeated for each ROW amd f2_values repeated for each COLUMN
grid_f1, grid_f2 = np.meshgrid(f1_values, f2_values)
#*NOTE - flatten 2d to 1d; then combine f1 and f2 into a single 2d array
flat_f1 = grid_f1.ravel()
flat_f2 = grid_f2.ravel()

grid_points = np.c_[flat_f1, flat_f2]

# !Do predictions, using GMM trained on phoneme 1, on custom grid

p_id = 1
#! Get params
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_1 = np.load(npy_filename, allow_pickle=True).item()

else:
    raise('File {} does not exist.'.format(npy_filename))

mean1 = GMM_parameters_phoneme_1['mu']
s1 = GMM_parameters_phoneme_1['s']
p1 = GMM_parameters_phoneme_1['p']



# !Do predictions, using GMM trained on phoneme 2, on custom grid

p_id = 2
#! Get params
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_2 = np.load(npy_filename, allow_pickle=True).item()

else:
    raise('File {} does not exist.'.format(npy_filename))

mean2 = GMM_parameters_phoneme_2['mu']
s2 = GMM_parameters_phoneme_2['s']
p2 = GMM_parameters_phoneme_2['p']

# ! Calculate likelihoods

likelihood_list_1 = np.zeros(len(grid_points))
likelihood_list_2 = np.zeros(len(grid_points))

#*NOTE - Loop through each gaussian; calc PDF of gaussian distributions (aka multivariate_normal) with mu and cov params; multiply by mixture weights ;then sum
for i in range(k):
    likelihood_1 = multivariate_normal.pdf(grid_points, mean=mean1[i], cov=s1[i])
    likelihood_list_1 = likelihood_list_1 + p1[i] * likelihood_1

    likelihood_2 = multivariate_normal.pdf(grid_points, mean=mean2[i], cov=s2[i])
    likelihood_list_2 = likelihood_list_2 + p2[i] * likelihood_2

# !Compare these predictions, to classify each point of the grid
# !Store these predictions in a 2D numpy array named "M", of shape N_f2 x N_f1 (the first dimension is f2 so that we keep f2 in the vertical axis of the plot)
# !M should contain "0.0" in the points that belong to phoneme 1 and "1.0" in the points that belong to phoneme 2



# !Boolean > 1D > 2D
binary_likelihoods = likelihood_list_1 < likelihood_list_2  # ^Boolean array
float_likelihoods = binary_likelihoods.astype(float)  # ^1D float array
M = float_likelihoods.reshape(grid_f1.shape)  # ^2D array




########################################/

################################################
#! Leave this untouched
# Visualize predictions on custom grid

# Create a figure
#fig = plt.figure()
fig, ax = plt.subplots()

# use aspect='auto' (default is 'equal'), to force the plotted image to be square, when dimensions are unequal
plt.imshow(M, aspect='auto')

# set label of x axis
ax.set_xlabel('f1')
# set label of y axis
ax.set_ylabel('f2')

# set limits of axes
plt.xlim((0, N_f1))
plt.ylim((0, N_f2))

# Set range and strings of ticks on axes
x_range = np.arange(0, N_f1, step=50)
x_strings = [str(x + min_f1) for x in x_range]
plt.xticks(x_range, x_strings)
y_range = np.arange(0, N_f2, step=200)
y_strings = [str(y + min_f2) for y in y_range]
plt.yticks(y_range, y_strings)

# set title of figure
title_string = 'Predictions on custom grid'
plt.title(title_string)

# add a colorbar
plt.colorbar()

N_samples = int(X.shape[0] / 2)
plt.scatter(X[:N_samples, 0] - min_f1, X[:N_samples, 1] - min_f2, marker='.', color='red', label='Phoneme 1')
plt.scatter(X[N_samples:, 0] - min_f1, X[N_samples:, 1] - min_f2, marker='.', color='green', label='Phoneme 2')

# add legend to the subplot
plt.legend()
plt.show()


### Decision Regions for $K=6$


In [ ]:

k = 6
# ! Combines for id 1 and 2
X = X_full[inds_1_and_2, :]
# !no. of samples
N = X.shape[0]
# !no. of features
D = X.shape[1]

min_f1 = int(np.min(X[:, 0]))
max_f1 = int(np.max(X[:, 0]))
min_f2 = int(np.min(X[:, 1]))
max_f2 = int(np.max(X[:, 1]))
N_f1 = max_f1 - min_f1
N_f2 = max_f2 - min_f2
print('f1 range: {}-{} | {} points'.format(min_f1, max_f1, N_f1))
print('f2 range: {}-{} | {} points'.format(min_f2, max_f2, N_f2))

#########################################
# Write your code here
# !Create a custom grid of shape N_f1 x N_f2
# !The grid will span all the values of (f1, f2) pairs, between [min_f1, max_f1] on f1 axis, and between [min_f2, max_f2] on f2 axis

f1_values = np.linspace(min_f1, max_f1, N_f1)
f2_values = np.linspace(min_f2, max_f2, N_f2)

#*NOTE -  get two 2d arrays from meshgrid; contains f1_values repeated for each ROW amd f2_values repeated for each COLUMN
grid_f1, grid_f2 = np.meshgrid(f1_values, f2_values)
#*NOTE - flatten 2d to 1d; then combine f1 and f2 into a single 2d array
flat_f1 = grid_f1.ravel()
flat_f2 = grid_f2.ravel()

grid_points = np.c_[flat_f1, flat_f2]

# !Do predictions, using GMM trained on phoneme 1, on custom grid

p_id = 1
#! Get params
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_1 = np.load(npy_filename, allow_pickle=True).item()

else:
    raise('File {} does not exist.'.format(npy_filename))

mean1 = GMM_parameters_phoneme_1['mu']
s1 = GMM_parameters_phoneme_1['s']
p1 = GMM_parameters_phoneme_1['p']



# !Do predictions, using GMM trained on phoneme 2, on custom grid

p_id = 2
#! Get params
npy_filename = 'data/GMM_params_phoneme_{:02}_k_{:02}.npy'.format(p_id, k)
if os.path.isfile(npy_filename):
    GMM_parameters_phoneme_2 = np.load(npy_filename, allow_pickle=True).item()

else:
    raise('File {} does not exist.'.format(npy_filename))

mean2 = GMM_parameters_phoneme_2['mu']
s2 = GMM_parameters_phoneme_2['s']
p2 = GMM_parameters_phoneme_2['p']

# ! Calculate likelihoods

likelihood_list_1 = np.zeros(len(grid_points))
likelihood_list_2 = np.zeros(len(grid_points))

#*NOTE - Loop through each gaussian; calc PDF of gaussian distributions (aka multivariate_normal) with mu and cov params; multiply by mixture weights ;then sum
for i in range(k):
    likelihood_1 = multivariate_normal.pdf(grid_points, mean=mean1[i], cov=s1[i])
    likelihood_list_1 = likelihood_list_1 + p1[i] * likelihood_1

    likelihood_2 = multivariate_normal.pdf(grid_points, mean=mean2[i], cov=s2[i])
    likelihood_list_2 = likelihood_list_2 + p2[i] * likelihood_2

# !Compare these predictions, to classify each point of the grid
# !Store these predictions in a 2D numpy array named "M", of shape N_f2 x N_f1 (the first dimension is f2 so that we keep f2 in the vertical axis of the plot)
# !M should contain "0.0" in the points that belong to phoneme 1 and "1.0" in the points that belong to phoneme 2



# !Boolean > 1D > 2D
binary_likelihoods = likelihood_list_1 < likelihood_list_2  # ^Boolean array
float_likelihoods = binary_likelihoods.astype(float)  # ^1D float array
M = float_likelihoods.reshape(grid_f1.shape)  # ^2D array




########################################/

################################################
#! Leave this untouched
# Visualize predictions on custom grid

# Create a figure
#fig = plt.figure()
fig, ax = plt.subplots()

# use aspect='auto' (default is 'equal'), to force the plotted image to be square, when dimensions are unequal
plt.imshow(M, aspect='auto')

# set label of x axis
ax.set_xlabel('f1')
# set label of y axis
ax.set_ylabel('f2')

# set limits of axes
plt.xlim((0, N_f1))
plt.ylim((0, N_f2))

# Set range and strings of ticks on axes
x_range = np.arange(0, N_f1, step=50)
x_strings = [str(x + min_f1) for x in x_range]
plt.xticks(x_range, x_strings)
y_range = np.arange(0, N_f2, step=200)
y_strings = [str(y + min_f2) for y in y_range]
plt.yticks(y_range, y_strings)

# set title of figure
title_string = 'Predictions on custom grid'
plt.title(title_string)

# add a colorbar
plt.colorbar()

N_samples = int(X.shape[0] / 2)
plt.scatter(X[:N_samples, 0] - min_f1, X[:N_samples, 1] - min_f2, marker='.', color='red', label='Phoneme 1')
plt.scatter(X[N_samples:, 0] - min_f1, X[N_samples:, 1] - min_f2, marker='.', color='green', label='Phoneme 2')

# add legend to the subplot
plt.legend()
plt.show()


## Three-Dimensional Feature Representation

A new feature matrix is created with three columns:

\begin{equation}
    X = [F1, F2, F1 + F2]
\end{equation}

This representation is used to demonstrate how linearly dependent features can affect covariance-based density models.


## Fitting a MoG Model to Linearly Dependent Features

This section attempts to fit a MoG model to the three-dimensional feature matrix containing $F1$, $F2$ and $F1+F2$.


### Singularity Issue

The third feature, $F1+F2$, is a linear combination of the first two features. This means it does not add independent information to the dataset.

Because the columns are linearly dependent, the covariance matrix can become singular or rank deficient. Gaussian likelihood calculations require the covariance matrix to be invertible, so a singular covariance matrix can cause numerical errors during model fitting.

This is a common issue when using covariance-based models with redundant or highly correlated features.


## Addressing Singular Covariance Matrices

Several approaches can be used to reduce or avoid singularity problems when fitting Gaussian models.


The main options are:

- **Covariance regularisation:** add a small value to the diagonal of each covariance matrix to make it invertible.
- **Feature removal:** remove the redundant $F1+F2$ feature, or remove either $F1$ or $F2$ if the combined feature is retained.
- **Noise injection:** add a very small amount of noise to the dependent feature to reduce exact linear dependence.
- **Constrained covariance structure:** use a diagonal or otherwise constrained covariance matrix to reduce sensitivity to feature dependence.

The implementation below uses covariance regularisation by adding a small multiple of the identity matrix to each covariance matrix. This improves numerical stability while only minimally changing the learned parameters.


In [ ]:
#########################################
# Write your code here



# ax.clear()
display.clear_output(wait=True)
p_id = 1  # id of the phoneme that will be used (e.g., 1 or 2)

k = 3  # Number of GMM components
X = np.zeros((len(f1), 3))
#########################################
# Write your code here
# !The shape of X is two-dimensional. Each row will represent a sample of the dataset, and each column will represent a feature (e.g., f1 or f2 or f1+f2)
# !Store f1 in the first column of X, f2 in the second column of X, and f1+f2 in the third column of X

X[:, 0] = f1.copy()  # F1
X[:, 1] = f2.copy()  #  F2
X[:, 2] = f1 + f2  # F1 + F2

# Filter X to contain only samples that belong to the chosen phoneme

X = X[phoneme_id == p_id, :]

#########################################
# !Plots
# !Before fit
fig = plt.figure()
ax1 = plt.axes(projection='3d')
title_string = 'Unfitted with Phoneme {}'.format(p_id)
print("Plotting data 3D")
plot_data_3D(X=X, title_string=title_string, ax=ax1)
plt.show()


# !For after fit
ax2 = plt.axes(projection='3d')
title_string = 'Fit with Phoneme {}'.format(p_id)
plot_data_3D(X=X, title_string=title_string, ax=ax2)

N, D = X.shape

p = np.ones((k)) / k

random_indices = np.floor(N * np.random.rand((k)))
random_indices = random_indices.astype(int)
mu = X[random_indices, :]  # ^Means (shape kxD)
s = np.zeros((k, D, D))  # ^shape kxDxD
n_iter = 150

# initialize covariances

for i in range(k):
    cov_matrix = np.cov(X.transpose())
    s[i, :, :] = cov_matrix / k  #

Z = np.zeros((N, k))  # shape Nxk

###############################
# run Expectation-Maximization algorithm for n_iter iterations
for t in range(n_iter):
    print('****************************************')
    print('Iteration {:03}/{:03}'.format(t + 1, n_iter))

    # Do the E-step
    Z = get_predictions(mu, s, p, X)  
    Z = normalize(Z, axis=1, norm='l1')  # ^Normalize

    # Do the M-step:
    for i in range(k):
        # Update means
        mu[i, :] = np.matmul(X.transpose(), Z[:, i]) / np.sum(Z[:, i])

        ###################################################
        # We will fit Gaussians with full covariance matrices
        mu_i = mu[i, :]
        mu_i = np.expand_dims(mu_i, axis=1)  # ^Expand dimensions
        mu_i_repeated = np.repeat(mu_i, N, axis=1)  # ^Repeat across samples

        term_1 = X.transpose() - mu_i_repeated
        term_2 = np.repeat(np.expand_dims(Z[:, i], axis=1), D, axis=1) * term_1.transpose()

        s[i, :, :] = np.matmul(term_1, term_2) / np.sum(Z[:, i])  # ^Update covariance

        #*NOTE - Regularization - add a constant to the diagonal leaving the others
        s[i, :, :] = s[i, :, :] + (1e-6 * np.eye(D))

        # ^Update weights
        p[i] = np.mean(Z[:, i])


print("Plotting Gaussians")

plot_gaussians(ax2, 2 * s, mu, X, 'Phoneme {}'.format(p_id))
plt.show()
#! Made prints pwetty
print('\nFinished.\n')

print("\nGaussian means (mu):\n")
print(f"\n{mu}\n")

print("\nGaussian covariances (s):\n")
print(f"\n{s}\n")

print("\nGuassian mixture weights (p):\n")
print(f"\n{p}\n")



